In [0]:
%run ../Configurations/Init_Scripts

In [0]:
from datetime import date
from datetime import timedelta

# Initialising variable

In [0]:
CreatedBy='InitialDataLoad'
database_name = "CS-CentralHub001"
table = "dbo.F_zm_COM1Cycle"
url = f"jdbc:sqlserver://{jdbcHostname}:{jdbcPort};databaseName={database_name}"
toDate=today = date.today()
fromDate=today - timedelta(days = 10)

print("Today's date:", today)
print("fromDate:", fromDate)
print("toDate:", toDate)



# Loading data from F_zm_COM1Cycle table

In [0]:
df_ULLogsSQL = spark.read.format("jdbc").option("url", url).option("dbtable", table).option("user", username).option("password", password).load().where("is_Legacy=0")
ULLogdf_Initial=(df_ULLogsSQL.withColumnRenamed('Rowkey','ULLogsUUID')
     .withColumn('SourceFilePath',concat("BlobContainerNm",lit("/"),"FileNameNm"))
     .withColumnRenamed('ActualFileNameNm','SourceFileName')
     .withColumn('CreatedBy',lit(CreatedBy))
     .withColumn('CreatedDt',col('RowCreatedTmstmp').cast("TIMESTAMP"))
     .withColumn('UpdatedBy',lit(CreatedBy))
     .withColumn('UpdatedDt',col('RowModifiedTmstmp').cast("TIMESTAMP"))
     .withColumn("CyclesLeft",col("CyclesLeft").cast("int"))
     .withColumn("BeginTime",col("BeginTime").cast("TIMESTAMP"))
     .withColumn("EndTime",col("EndTime").cast("TIMESTAMP"))
     .withColumn("SBC_sn",upper(col("SBC_sn")))
     )
AzureIOT=(ULLogdf_Initial.select('CycleID',
                                'SBC_sn',
                                'EZCard',
                                'CyclesLeft',
                                'BeginTime',
                                'EndTime',
                                "SourceFilePath",
                                "SourceFileName",
                                'CreatedBy',
                                'CreatedDt',
                                'UpdatedBy',
                                'UpdatedDt',
                                'is_Legacy')
                                # .filter("createddt<'{0}'".format(today))
                                )
AzureIOT=AzureIOT.persist()
AzureIOT.createOrReplaceTempView("AzureIOT")
print("AzureIOTCount: "+str(AzureIOT.count()))     

# Loading data from silverzone.factullogs_com1 table

In [0]:
DatabricksData=(spark.sql("SELECT * FROM silverzone.factullogs_com1").select('CycleID','SBC_sn','EZCard','CyclesLeft','BeginTime','EndTime','SourceFilePath','SourceFileName','CreatedBy','CreatedDt','UpdatedBy','UpdatedDt')
# .filter("createddt<'{0}'".format(today))
)
DatabricksData.createOrReplaceTempView("DatabricksData")
print("DatabricksDataCount: "+str(DatabricksData.count()))

# Cycle comparision using BeginTime

In [0]:
AzureIOTCycleCount=spark.sql("""select to_date(begintime, 'yyyy-MM-dd') as Begintime , count(1) as AzureProd_CycleCount from AzureIOT where to_date(begintime, 'yyyy-MM-dd')>='{0}' and to_date(Begintime, 'yyyy-MM-dd') <='{1}' group by to_date(begintime, 'yyyy-MM-dd')""".format(fromDate,toDate))
DBCycleCount=spark.sql("""select to_date(begintime, 'yyyy-MM-dd') as Begintime , count(1) as DBProd_CycleCount from DatabricksData where where to_date(begintime, 'yyyy-MM-dd')>='{0}' and to_date(Begintime, 'yyyy-MM-dd') <='{1}' group by to_date(begintime, 'yyyy-MM-dd')""".format(fromDate,toDate))
CycleDifferance=AzureIOTCycleCount.join(DBCycleCount,['BeginTime'],'Outer').withColumn("countDifference",col('AzureProd_CycleCount')-col('DBProd_CycleCount')).orderBy(col("countDifference").desc())
CycleDifferance.display()

# Cycle comparision using BeginTime,Ezcard,Cyclesleft

In [0]:
AzureCycleCount1=spark.sql("""select to_date(begintime, 'yyyy-MM-dd') as Begintime,EZCard,cyclesleft, count(1) as AzureProd_CycleCount 
                                from  AzureIOT
                                 where to_date(begintime, 'yyyy-MM-dd')>='{0}' and to_date(Begintime, 'yyyy-MM-dd') <='{1}'
                                group by to_date(begintime, 'yyyy-MM-dd'),cyclesleft,EZCard
""".format(fromDate,toDate))
DBCycleCount1=spark.sql("""select to_date(begintime, 'yyyy-MM-dd') as Begintime, EZCard,cyclesleft, count(1) as DBProd_CycleCount 
                            from DatabricksData 
                             where to_date(begintime, 'yyyy-MM-dd')>='{0}' and to_date(Begintime, 'yyyy-MM-dd') <='{1}'
                            group by to_date(begintime, 'yyyy-MM-dd'),cyclesleft,EZCard
""".format(fromDate,toDate))

CycleDifferance1=(AzureCycleCount1.join(DBCycleCount1,['BeginTime','cyclesleft','EZCard'],'Outer')
                                    .withColumn("countDifference",expr("coalesce(AzureProd_CycleCount,0)") - expr("coalesce(DBProd_CycleCount,0)"))
                                    .orderBy(col("countDifference").desc())
                )
CyclesCountNotPresentINProduction=CycleDifferance1.filter('countDifference<0').groupBy("begintime").agg(count("cyclesleft").alias("missingInAzureProd"))
CyclesCountNotPresentINDatabricks=CycleDifferance1.filter('countDifference>0').groupBy("begintime").agg(count("cyclesleft").alias("missingInDBProd"))      
CyclesCountMatching=CycleDifferance1.filter('countDifference=0').groupBy("begintime").agg(count("cyclesleft").alias("MatchingCount"))  
Finaltable=(CyclesCountNotPresentINProduction.join(CyclesCountNotPresentINDatabricks,['begintime'],'outer')
                                             .join(CyclesCountMatching,['begintime'],'outer')
                                             .join(CycleDifferance,['begintime'],'outer')
                                             .withColumn("ExpectedCount",coalesce(col("missingInAzureProd"),lit(0))+coalesce(col("missingInDBProd"),lit(0))+coalesce(col("MatchingCount"),lit(0)))
                                             .withColumn("Comment",when(((col("ExpectedCount")==col("AzureProd_CycleCount")) & (col("ExpectedCount")==col("DBProd_CycleCount"))),'Cycles Count is good')
                                             .when(((col("ExpectedCount")==col("AzureProd_CycleCount"))),'Prod Cycles Count is good')
                                             .when(( (col("ExpectedCount")==col("DBProd_CycleCount"))),'QA Cycles Count is good')
                                             .otherwise("QA and Prod not Good")
                                             )
                                             
)
Finaltable.select("Begintime","AzureProd_CycleCount","DBProd_CycleCount",expr("coalesce(missingInAzureProd,0) as MissingInAzureProd "),expr("coalesce(missingInDBProd,0) as MissingInDBProd "),"MatchingCount","ExpectedCount","Comment").display()


# Change BegintDate variable Accordingy
## All cycles greater then or equal to Begintime will be analysed

In [0]:
BeginDate='2023-05-01'

In [0]:
AzureIOTCycleCount1=spark.sql("""select to_date(begintime, 'yyyy-MM-dd') as Begintime,EZCard,cyclesleft, count(1) as AzureProd_CycleCount 
                                from AzureIOT 
                                where to_date(begintime, 'yyyy-MM-dd')>='{0}' 
                                group by to_date(begintime, 'yyyy-MM-dd'),cyclesleft,EZCard
""".format(BeginDate))
DBCycleCount1=spark.sql("""select to_date(begintime, 'yyyy-MM-dd') as Begintime, EZCard,cyclesleft, count(1) as DBProd_CycleCount 
                            from DatabricksData 
                            where to_date(begintime, 'yyyy-MM-dd')>='{0}' 
                            group by to_date(begintime, 'yyyy-MM-dd'),cyclesleft,EZCard
""".format(BeginDate))
CycleDifferance1=(AzureIOTCycleCount1.join(DBCycleCount1,['BeginTime','cyclesleft','EZCard'],'Outer')
                                    .withColumn("countDifference",expr("coalesce(AzureProd_CycleCount,0)") - expr("coalesce(DBProd_CycleCount,0)"))
                                    .filter("countDifference !=0 or countDifference is null")
                                    .orderBy(col("countDifference").desc())
                )
CyclesCountNotPresentINProduction=CycleDifferance1.filter(' countDifference<0').count() 
CyclesCountNotPresentINDatabricks=CycleDifferance1.filter(' countDifference>0').count()              
print("CyclesCountNotPresentINProduction:" + str(CyclesCountNotPresentINProduction))
print("CyclesCountNotPresentINDatabricks:" + str(CyclesCountNotPresentINDatabricks))

In [0]:
CyclemissingInDatabricks=CycleDifferance1.filter('countDifference>0')
print("CyclemissingInDatabricks:" + str(CyclemissingInDatabricks.count()))
joinedWithmissingData=AzureIOT.join(CyclemissingInDatabricks,['cyclesleft','EZCard'],'inner').filter("createddt<'{0}'".format(today))
joinedWithmissingData.display()
missingfilesDatabricks=joinedWithmissingData.groupBy("SourceFilePath","SourceFileName").count()
missingfilesDatabricks.display()

In [0]:

logsourcefilesprocessed=spark.sql("select * from silverzone.logsourcefilesprocessed where configid=25")
logsourcefilesprocessed.join(missingfilesDatabricks,['SourceFileName'],'inner').display()

In [0]:
cycleMissingInProd=CycleDifferance1.filter('countDifference<0')
print("cycleMissingInProd:" + str(cycleMissingInProd.count()))
joinedWithmissingDataprod=DatabricksData.join(cycleMissingInProd,['cyclesleft','EZCard'],'inner').filter("createddt<'{0}'".format(today))
joinedWithmissingDataprod.display()
missingfilesProd=joinedWithmissingDataprod.groupBy("SourceFilePath","SourceFileName").count()
missingfilesProd.display()

# For Manual Analysis

In [0]:
%sql
select * from silverzone.factullogs_com1  
where 
--SourceFilename='COM1_3111 0278 4546 A521_UL_7_2023_0426_001829.csv' 
Ezcard='PPD2021229ILA2470'
and CyclesLeft='15'

In [0]:
%sql
select * from AzureIOT
where 
-- SourceFilename='COM1_3111 0278 4546 A521_UL_7_2023_0426_001829.csv' 
Ezcard='PPD2021229ILA2470'
and CyclesLeft='15'